In [28]:
from collections import defaultdict
import numpy as np

class ItemKNNRecommender:
    def __init__(self, k=40, shrinkage=25.0, min_candidate_popularity=5):
        self.k = k
        self.shrinkage = shrinkage
        self.min_candidate_popularity = min_candidate_popularity

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.ratings = np.full((n_users, n_items), np.nan, dtype=np.float32)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        for user_id, item_id, rating, _ in rows:
            uidx = self.user_to_idx[user_id]
            iidx = self.item_to_idx[item_id]
            self.ratings[uidx, iidx] = rating
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
        self.global_mean = float(np.nanmean(self.ratings))
        self.user_means = np.nanmean(self.ratings, axis=1)
        self.item_means = np.nanmean(self.ratings, axis=0)
        self.user_means = np.where(np.isnan(self.user_means), self.global_mean, self.user_means)
        self.item_means = np.where(np.isnan(self.item_means), self.global_mean, self.item_means)
        mask = ~np.isnan(self.ratings)
        centered = np.where(mask, self.ratings - self.user_means[:, None], 0.0)
        numerator = centered.T @ centered
        norms = np.sqrt(np.sum(centered * centered, axis=0))
        denominator = norms[:, None] * norms[None, :]
        similarity = np.divide(numerator, denominator,
            out=np.zeros_like(numerator, dtype=np.float32), where=denominator > 0)
        co_counts = mask.astype(np.float32).T @ mask.astype(np.float32)
        significance = co_counts / (co_counts + self.shrinkage)
        self.similarity = similarity * significance
        np.fill_diagonal(self.similarity, 0.0)
        self.neighbors = {}
        for item_idx in range(n_items):
            sims = self.similarity[item_idx]
            order = np.argsort(np.abs(sims))[::-1]
            order = [idx for idx in order if sims[idx] != 0.0][: self.k]
            self.neighbors[item_idx] = order
        return self

    def predict(self, user_id, item_id):
        if user_id not in self.user_to_idx:
            return self._clip(self.item_mean(item_id))
        if item_id not in self.item_to_idx:
            return self._clip(self.user_mean(user_id))
        uidx = self.user_to_idx[user_id]
        target_idx = self.item_to_idx[item_id]
        baseline = self.user_means[uidx]
        numerator = denominator = 0.0
        for neighbor_idx in self.neighbors[target_idx]:
            rating = self.ratings[uidx, neighbor_idx]
            if np.isnan(rating):
                continue
            sim = float(self.similarity[target_idx, neighbor_idx])
            numerator += sim * (float(rating) - baseline)
            denominator += abs(sim)
        if denominator == 0.0:
            return self._clip(0.6 * baseline + 0.4 * self.item_means[target_idx])
        return self._clip(baseline + numerator / denominator)

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n):
        return [(item_id, self.item_mean(item_id))
            for item_id, _ in sorted(self.item_popularity.items(),
                key=lambda x: (x[1], self.item_mean(x[0])), reverse=True)[:top_n]]

    def user_mean(self, user_id):
        if user_id not in self.user_to_idx:
            return self.global_mean
        return float(self.user_means[self.user_to_idx[user_id]])

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return float(self.item_means[self.item_to_idx[item_id]])

    def _clip(self, value):
        return min(5.0, max(1.0, float(value)))


class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=20,
                 min_candidate_popularity=5, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        rating_sum = rating_count = 0
        for user_id, item_id, rating, _ in rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
            rating_count += 1
        self.global_mean = rating_sum / rating_count if rating_count > 0 else 3.0
        rng = np.random.default_rng(self.seed)
        self.U = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)
        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            sq_err = 0.0
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                        + float(self.U[uidx] @ self.V[iidx]))
                err = rating - pred
                sq_err += err ** 2
                u_old = self.U[uidx].copy()
                self.U[uidx] += self.lr * (err * self.V[iidx] - self.reg * self.U[uidx])
                self.V[iidx] += self.lr * (err * u_old - self.reg * self.V[iidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i:
            return self._clip(self.global_mean)
        if not has_u:
            return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i:
            return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx = self.user_to_idx[user_id]
        iidx = self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                          + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(),
            key=lambda item_id: (self.item_popularity[item_id], self.item_mean(item_id)), reverse=True)
        return [(item_id, self.item_mean(item_id)) for item_id in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

    def _clip(self, value):
        return min(5.0, max(1.0, float(value)))


class EnsembleRecommender:
    def __init__(self, knn_model, mf_model, weight_knn=0.5, min_candidate_popularity=5):
        self.knn = knn_model
        self.mf = mf_model
        self.w_knn = weight_knn
        self.w_mf = 1.0 - weight_knn
        self.min_candidate_popularity = min_candidate_popularity
        self.items = sorted(set(knn_model.items) | set(mf_model.items))
        self.item_popularity = mf_model.item_popularity
        self.user_rated_items = defaultdict(set)
        for uid, rated in knn_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        for uid, rated in mf_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        self.users = sorted(set(knn_model.users) | set(mf_model.users))
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}

    def predict(self, user_id, item_id):
        return min(5.0, max(1.0, self.w_knn * self.knn.predict(user_id, item_id)
                                + self.w_mf  * self.mf.predict(user_id, item_id)))

    def recommend(self, user_id, top_n=10):
        seen = self.user_rated_items.get(user_id, set())
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

In [ ]:
import argparse
import math
import pickle
from collections import defaultdict
from pathlib import Path


def read_ratings(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split("\t")
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows


def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        sq_errors.append((rating - pred) ** 2)
        abs_errors.append(abs(rating - pred))
    return {
        "rmse": math.sqrt(sum(sq_errors) / len(sq_errors)),
        "mae": sum(abs_errors) / len(abs_errors),
    }


def topn_metrics(model, eval_rows, top_n=10, relevant_threshold=4.0):
    known_users = getattr(model, "user_to_idx", {})
    eval_users = sorted({u for u, _, _, _ in eval_rows if u in known_users})
    relevant_by_user = defaultdict(set)
    for user_id, item_id, rating, _ in eval_rows:
        if rating >= relevant_threshold:
            relevant_by_user[user_id].add(item_id)
    all_recommended = set()
    novelty_scores = []
    precisions, recalls = [], []
    total_interactions = sum(model.item_popularity.values())
    for user_id in eval_users:
        recs = model.recommend(user_id, top_n=top_n)
        rec_items = [item_id for item_id, _ in recs]
        all_recommended.update(rec_items)
        for item_id in rec_items:
            pop = model.item_popularity.get(item_id, 1)
            novelty_scores.append(-math.log2(pop / total_interactions))
        relevant = relevant_by_user.get(user_id, set())
        if relevant:
            hits = len(set(rec_items) & relevant)
            precisions.append(hits / top_n)
            recalls.append(hits / len(relevant))
    precision = sum(precisions) / len(precisions) if precisions else 0.0
    recall = sum(recalls) / len(recalls) if recalls else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "coverage": len(all_recommended) / max(len(model.items), 1),
        "novelty": sum(novelty_scores) / len(novelty_scores) if novelty_scores else 0.0,
        "users_evaluated": len(eval_users),
    }


def print_metrics_table(results, top_n):
    col_w = 14
    header = f"{'지표':<22}" + "".join(f"{name:>{col_w}}" for name in results)
    sep = "-" * len(header)
    print(sep)
    print(header)
    print(sep)
    metric_labels = [
        ("rmse",      "RMSE (↓)"),
        ("mae",       "MAE (↓)"),
        ("precision", f"Precision@{top_n} (↑)"),
        ("recall",    f"Recall@{top_n} (↑)"),
        ("f1",        f"F1@{top_n} (↑)"),
        ("coverage",  "Coverage (↑)"),
        ("novelty",   "Novelty (↑)"),
    ]
    combined = {}
    for name, m in results.items():
        combined[name] = {**m["rating"], **m["topn"]}
    for key, label in metric_labels:
        row = f"{label:<22}"
        for name in results:
            val = combined[name].get(key, float("nan"))
            row += f"{val:>{col_w}.4f}"
        print(row)
    print(sep)
    for name in results:
        print(f"  [{name}] 평가 유저 수: {results[name]['topn']['users_evaluated']}")
    print(sep)


def print_recommendations(model, titles, user_id, top_n, model_name):
    print(f"\n[{model_name}] 유저 {user_id} 추천 Top-{top_n}")
    print("-" * 55)
    for rank, (item_id, score) in enumerate(model.recommend(user_id, top_n=top_n), start=1):
        title = titles.get(item_id, f"movieId={item_id}")
        print(f"  {rank:>2}. {title:<40} | 예측 평점={score:.3f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="ml-100k")
    parser.add_argument("--model-path", default="saved_models/models.pkl")
    parser.add_argument("--top-n", type=int, default=10)
    parser.add_argument("--sample-user", type=int, default=1)
    args = parser.parse_args(args=[])

    with open(args.model_path, "rb") as f:
        bundle = pickle.load(f)

    knn_model      = bundle["knn"]
    mf_model       = bundle["mf"]
    ensemble_model = bundle["ensemble"]
    titles         = bundle["titles"]

    test_rows = read_ratings(Path(args.data_dir) / "ua.test")

    print("=" * 60)
    print("튜닝 결과 요약 (validation 기준)")
    print("=" * 60)

    print("\n[KNN] K별 validation 성능")
    for k, rmse, mae in bundle["knn_validation_results"]:
        mark = " ← 선택" if k == bundle["best_k"] else ""
        print(f"  K={k:>3}: RMSE={rmse:.4f}, MAE={mae:.4f}{mark}")

    print("\n[MF] n_factors별 validation 성능")
    for nf, rmse, mae in bundle["mf_validation_results"]:
        mark = " ← 선택" if nf == bundle["best_factors"] else ""
        print(f"  factors={nf:>3}: RMSE={rmse:.4f}, MAE={mae:.4f}{mark}")

    print("\n[Ensemble] 가중치별 validation 성능")
    for w_knn, rmse, mae in bundle["ensemble_weight_results"]:
        mark = " ← 선택" if abs(w_knn - bundle["best_w_knn"]) < 1e-6 else ""
        print(f"  w_knn={w_knn:.1f} w_mf={1-w_knn:.1f}: RMSE={rmse:.4f}, MAE={mae:.4f}{mark}")

    print("\n" + "=" * 60)
    print(f"최종 평가 (ua.test, Top-{args.top_n})")
    print("=" * 60)

    models = {
        "ItemKNN":  knn_model,
        "MF":       mf_model,
        "Ensemble": ensemble_model,
    }
    results = {}
    for name, model in models.items():
        r_m = rating_metrics(model, test_rows)
        t_m = topn_metrics(model, test_rows, top_n=args.top_n)
        results[name] = {"rating": r_m, "topn": t_m}

    print()
    print_metrics_table(results, args.top_n)

    print("=" * 60)
    print(f"샘플 추천 결과 (유저 ID={args.sample_user})")
    print("=" * 60)

    for name, model in models.items():
        print_recommendations(model, titles, args.sample_user, args.top_n, name)

    print()


if __name__ == "__main__":
    main()

main()

튜닝 결과 요약 (validation 기준)

[KNN] K별 validation 성능
  K= 20: RMSE=0.9772, MAE=0.7498
  K= 40: RMSE=0.9588, MAE=0.7397
  K= 80: RMSE=0.9458, MAE=0.7335 ← 선택

[MF] n_factors별 validation 성능
  factors= 10: RMSE=0.9467, MAE=0.7447
  factors= 20: RMSE=0.9466, MAE=0.7447
  factors= 50: RMSE=0.9454, MAE=0.7437 ← 선택

[Ensemble] 가중치별 validation 성능
  w_knn=0.0 w_mf=1.0: RMSE=0.9454, MAE=0.7437
  w_knn=0.1 w_mf=0.9: RMSE=0.9392, MAE=0.7384
  w_knn=0.2 w_mf=0.8: RMSE=0.9343, MAE=0.7340
  w_knn=0.3 w_mf=0.7: RMSE=0.9308, MAE=0.7304
  w_knn=0.4 w_mf=0.6: RMSE=0.9287, MAE=0.7279
  w_knn=0.5 w_mf=0.5: RMSE=0.9281, MAE=0.7262 ← 선택
  w_knn=0.6 w_mf=0.4: RMSE=0.9288, MAE=0.7255
  w_knn=0.7 w_mf=0.3: RMSE=0.9310, MAE=0.7259
  w_knn=0.8 w_mf=0.2: RMSE=0.9345, MAE=0.7273
  w_knn=0.9 w_mf=0.1: RMSE=0.9395, MAE=0.7299
  w_knn=1.0 w_mf=0.0: RMSE=0.9458, MAE=0.7335

최종 평가 (ua.test, Top-10)
